# Deep Learning Fundamentals: Your First Neural Network with PyTorch

In this notebook, you will build and train your first **neural network** using **PyTorch**.

#### Learning objectives

By the end of this notebook, you will be able to:
- Load and preprocess data for neural network training
- Define a simple feedforward neural network architecture in PyTorch
- Understand forward propagation through the network
- Train a neural network using backpropagation and gradient descent
- Evaluate model performance and visualize training progress
- Experiment with different activation functions and architectures

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("notebook")

# Check for GPU availability and set device
device = torch.device(
    "cuda" if torch.cuda.is_available() # Check for CUDA (NVIDIA GPU)
    else "mps" if torch.backends.mps.is_available() # Check for Apple Silicon GPU (Metal Performance Shaders)
    else "cpu" # Default to CPU if no GPU is available
    )

print("Imports ready.")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

## 1. Generate and explore the dataset

We'll use a synthetic **moons dataset** - a classic non-linear classification problem. This dataset creates two interleaving half circles, which cannot be separated by a simple linear classifier.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Why would a linear classifier struggle with this dataset?
- How many samples and features do we have?
- Is the dataset balanced (equal number of samples per class)?
</div>

In [ ]:
# Create dataset
X, y = make_moons(n_samples=1000, noise=0.4, random_state=42)

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"X shape: {X.shape} | y shape: {y.shape}")
print(f"Number of samples per class: Class 0: {(y==0).sum()}, Class 1: {(y==1).sum()}")

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', edgecolor='k', s=50, alpha=0.7)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Moons Dataset: Two-Class Non-Linear Classification')
plt.tight_layout()
plt.show()

PyTorch works with tensors, which are similar to NumPy arrays but optimized for deep learning tasks. We'll convert our data into PyTorch tensors before feeding it into the neural network.

In [ ]:
# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

X_test = _
y_test = _

## 2. Build a Simple Neural Network

Now we'll define our neural network architecture. We'll create a simple feedforward network with:
- **Input layer**: 2 features (x, y coordinates)
- **Hidden layer 1**: 16 neurons with ReLU activation
- **Hidden layer 2**: 8 neurons with ReLU activation
- **Output layer**: 2 neurons

Without going into too much technical detail, to create our neural network we will define a class that inherits from `torch.nn.Module` (this means we will create a custom class that has the same properties as `torch.nn.Module`, which is the base class for all neural network models). This class will contain the layers of our network and the forward pass logic. In the initialization method (`__init__`), we will define the layers of our network. In the `forward` method, we will specify how data flows through the network.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Why do we use activation functions between layers?
- What would happen if we removed all activation functions?
- Why does the output layer have 2 neurons for binary classification? How many neurons would we need for a 3-class classification problem?
</div>

In [ ]:
class SimpleNeuralNetwork(nn.Module):
    def __init__(self):
        super(SimpleNeuralNetwork, self).__init__()
        
        # Define layers
        self.fc1 = nn.Linear(2, 16) # First hidden layer
        self.relu1 = nn.ReLU()      # Activation function
        self.fc2 = nn.Linear(16, 8) # Second hidden layer
        self.relu2 = nn.ReLU()      # Activation function
        self.fc3 = nn.Linear(8, 2)  # Output layer
    
    def forward(self, x):
        """Forward pass through the network"""
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        return out

# Initialize the model
model = SimpleNeuralNetwork()

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")

## 3. Understand the Initialized Model

Now we have defined our neural network architecture! However, the coefficients (weights and biases) are still **randomly initialized**. 

Let's see what happens when we use this untrained model to make predictions. Since the weights are random, we expect the accuracy to be close to random guessing (around 50% for a binary classification problem).

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Which steps in the training process will help the model learn better weights and improve accuracy?
- If the model makes a mistake during training, how does it adjust its weights to learn from that mistake?
</div>

In [ ]:
with torch.no_grad():  # Disable gradient computation for evaluation (for computational efficiency)

    # Training data
    train_outputs = model(X_train) # Forward pass
    _, train_predicted = torch.max(train_outputs, 1)  # Get class with highest score
    train_accuracy = accuracy_score(y_train.numpy(), train_predicted.numpy())
    
    # Test data
    test_outputs = model(X_test)
    _, test_predicted = torch.max(test_outputs, 1)
    test_accuracy = accuracy_score(y_test.numpy(), test_predicted.numpy())

print(f"Training Accuracy (untrained): {train_accuracy * 100:.2f}%")
print(f"Test Accuracy (untrained): {test_accuracy * 100:.2f}%")

## 4. Train the Neural Network

Training a neural network involves several key steps:
1. Forward pass
2. Compute loss
3. Backward pass (compute gradients)
4. Update weights

Before we can proceed with the training, we have to define our **loss function** and **optimizer**:

- **Loss function (Cross-Entropy)**: Measures how well our predictions match the true labels. Lower loss = better predictions.
- **Optimizer (Adam)**: Determines how we update the network weights to minimize the loss. Adam is a popular choice because it adapts the learning rate automatically.

Think of the loss function as a GPS showing how far you are from your destination, and the optimizer as the navigation system deciding which direction to go.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- What does the learning rate control?
- What happens if the learning rate is too high? Too low?
</div>

In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
learning_rate = 0.01
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"Loss function: {criterion}")
print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Learning rate: {learning_rate}")

Now we can train the model!

During training, we will perform multiple iterations (epochs) over the training data. During each epoch the model weights are updated based on the computed gradients from the loss, and the model learns to make better predictions. After each epoch, we will evaluate the model on both the training and test data to monitor its performance.

This process repeats for many epochs until the model learns the patterns in the data.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Why do we need `optimizer.zero_grad()`? (Hint: gradients accumulate by default in PyTorch)
- What does `loss.backward()` do?
- What is an "epoch"? Why do we need multiple epochs to train a model? Is this similar to how we train machine learning models like logistic regression?
</div>

In [ ]:
# Training parameters
num_epochs = 100
training_loss = []
test_loss = []
training_accuracy = []
test_accuracy = []

print("Starting training...")
print("=" * 50)

for epoch in range(num_epochs):
    # ===== TRAINING STEP =====
    # Set model to training mode
    model.train()
    
    # Forward pass: compute predicted outputs (on training data)
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    
    # Backward pass: compute gradients
    optimizer.zero_grad()  # Why do we need to zero the gradients before calling backward()?
    loss.backward()        # Compute new gradients
    
    # Update weights
    optimizer.step()       # Apply gradient descent (this updates the model parameters)
    
    # ===== EVALUATION STEP =====
    model.eval()  # Set model to evaluation mode
    with torch.no_grad():  # Disable gradient computation
        # Evaluate on training data
        train_outputs = model(X_train)
        train_loss_value = criterion(train_outputs, y_train)

        training_loss.append(train_loss_value.item())
        _, train_pred = torch.max(train_outputs, 1)
        training_accuracy.append(accuracy_score(y_train.numpy(), train_pred.numpy()))
        
        # Evaluate on test data
        test_outputs = model(X_test)
        test_loss_value = criterion(test_outputs, y_test)
        
        test_loss.append(test_loss_value.item())
        _, test_pred = torch.max(test_outputs, 1)
        test_accuracy.append(accuracy_score(y_test.numpy(), test_pred.numpy()))
    
    # Print progress every 20 epochs
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] | "
              f"Train Loss: {training_loss[-1]:.4f} | "
              f"Test Loss: {test_loss[-1]:.4f} | "
              f"Train Acc: {training_accuracy[-1]*100:.2f}% | "
              f"Test Acc: {test_accuracy[-1]*100:.2f}%")

print("=" * 50)
print("Training complete!\n")

# Print the final results
print(f"Final Results:")
print(f"Training Loss: {training_loss[-1]:.4f}")
print(f"Test Loss: {test_loss[-1]:.4f}")
print(f"Training Accuracy: {training_accuracy[-1] * 100:.2f}%")
print(f"Test Accuracy: {test_accuracy[-1] * 100:.2f}%")

## 5. Visualize Training Progress

Let's plot the training and test loss/accuracy over epochs to see how the model learned.

Watch how the loss decreases and accuracy increases as the network learns the patterns in the data. The gap between training and test metrics can indicate overfitting.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Does the loss decrease over time?
- Is there any sign of overfitting (training accuracy much higher than test accuracy)?
- When did the model converge (stop improving significantly)?
- What do you notice about the test loss curve? Is it smooth or noisy?
</div>

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot loss
ax1.plot(training_loss, label='Train Loss', linewidth=2, color='#2E86AB')
ax1.plot(test_loss, label='Test Loss', linewidth=2, color='#A23B72')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Test Loss Over Time', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Plot accuracy
ax2.plot([acc*100 for acc in training_accuracy], label='Train Accuracy', linewidth=2, color='#2E86AB')
ax2.plot([acc*100 for acc in test_accuracy], label='Test Accuracy', linewidth=2, color='#A23B72')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Training and Test Accuracy Over Time', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 105])

plt.tight_layout()
plt.show()

## 6. Evaluate the Model

Now we can evaluate the final performance of our trained model on the test set. We will compute the final loss and accuracy, and plot the confusion matrix to see how well the model classified the samples.

In [ ]:
# Make predictions on test set
model.eval()
with torch.no_grad():
    outputs = model(X_test)
    _, y_pred = torch.max(outputs, 1)
    y_pred_np = y_pred.numpy()

# Calculate accuracy
test_accuracy = accuracy_score(y_test, y_pred_np)
print(f"Final Test Accuracy: {test_accuracy * 100:.2f}%")

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred_np)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Class 0', 'Class 1'])
disp.plot(cmap='Blues', values_format='d', colorbar=False)
plt.grid(False)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

### Decision Boundary Visualization

Let's visualize how the neural network separates the two classes in the feature space.

The colored background shows which class the model predicts for each point in space. Notice how the boundary is curved and complex - this is the power of neural networks with non-linear activation functions!

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Is the decision boundary linear or non-linear?
- How does it compare to what a linear classifier (like logistic regression) could achieve?
- Can you see why this is called a "decision boundary"?
- Does the boundary capture the moon shape well?
</div>

In [ ]:
def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    """Plot decision boundary of the neural network"""
    # Create a mesh
    x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
    y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    
    # Make predictions
    with torch.no_grad():
        outputs = model(grid)
        _, predictions = torch.max(outputs, 1)
        Z = predictions.numpy().reshape(xx.shape)
        
    # Plot
    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', edgecolor='k', s=50, alpha=0.8)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title(title)
    plt.show()

# Plot decision boundary on test set
plot_decision_boundary(model, X, y, title="Neural Network Decision Boundary (Test Set)")

## Key Takeaways

Congratulations! You've built and trained your first neural network. Here's what we covered:

### Neural Network Basics:
- Neural networks are composed of **layers of neurons** connected by **weights**
- **Activation functions** (like ReLU) introduce non-linearity, allowing networks to learn complex patterns
- Without activation functions, a neural network would just be a fancy linear model!

### Training Process (The Learning Cycle):
1. **Forward pass**: Input → Hidden layers → Output (make predictions)
2. **Compute loss**: Measure how wrong the predictions are
3. **Backward pass**: Calculate gradients using backpropagation (how to improve)
4. **Update weights**: Use optimizer (e.g., Adam) to adjust parameters

### What Makes Neural Networks Powerful:
- Can learn **non-linear** decision boundaries
- Automatically **extract features** from data
- Scale to complex problems with enough data and compute

**Next steps**: Try the optional exercises below to deepen your understanding!

***
## Optional exercises (for early finishers)
### A. Build Your Own Network

Now it's your turn! Try modifying the neural network architecture and see how it affects performance.

1. Create a deeper network with 3 hidden layers (sizes: 32, 16, 8)
2. Train it for 100 epochs
3. Compare its performance to the original 2-layer network
4. Visualize the decision boundary

<div class="alert alert-block alert-info">

**Bonus challenges:**
- Try different learning rates (0.001, 0.01, 0.1)
- Experiment with different hidden layer sizes
- Try different activation functions (e.g. Tanh, Sigmoid, Leaky ReLU, etc.)
</div>

In [ ]:
class CustomNeuralNetwork(nn.Module):
    def __init__(self):
        super(CustomNeuralNetwork, self).__init__()
        # Define your architecture here
        # Example: 3 hidden layers with 32, 16, 8 neurons
        # self.fc1 = nn.Linear(2, 32)
        # self.relu1 = nn.ReLU()
        # ... add more layers ...
        pass
    
    def forward(self, x):
        # Define forward pass here
        # out = self.fc1(x)
        # out = self.relu1(out)
        # ... pass through remaining layers ...
        pass

your_custom_model = CustomNeuralNetwork()
print(your_custom_model)

# TODO: Define loss and optimizer
criterion = ...
optimizer = ...

# TODO: Train your model (copy training loop from above and modify)

# TODO: Evaluate the trained model and compare with original model - which performed better? Why?
